In [1]:
import pygame
import random

pygame.init()

WIDTH, HEIGHT = 300, 600
BLOCK_SIZE = 30
TOP_MARGIN = 50  


screen = pygame.display.set_mode((WIDTH, HEIGHT + TOP_MARGIN))
pygame.display.set_caption("KMB-tetris")


BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GRAY = (40, 40, 40)
COLORS = [
    (0, 255, 255), (255, 255, 0), (128, 0, 128),
    (0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 165, 0)
]


SHAPES = [
    [[1, 1, 1, 1]],
    [[1, 1], [1, 1]],
    [[0, 1, 0], [1, 1, 1]],
    [[1, 1, 0], [0, 1, 1]],
    [[0, 1, 1], [1, 1, 0]],
    [[1, 0, 0], [1, 1, 1]],
    [[0, 0, 1], [1, 1, 1]]
]

class Tetris:
    def __init__(self):
        self.board = [[0] * (WIDTH // BLOCK_SIZE) for _ in range(HEIGHT // BLOCK_SIZE)]
        self.game_over = False
        self.score = 0  
        self.new_piece()

    def new_piece(self):
        self.shape = random.choice(SHAPES)
        self.color = random.randint(1, len(COLORS))
        self.x = (WIDTH // BLOCK_SIZE) // 2 - len(self.shape[0]) // 2
        self.y = 0

        if self.check_collision():
            self.game_over = True

    def check_collision(self, dx=0, dy=0, shape=None):
        if shape is None:
            shape = self.shape
        for y, row in enumerate(shape):
            for x, cell in enumerate(row):
                if cell:
                    nx, ny = self.x + x + dx, self.y + y + dy
                    if nx < 0 or nx >= WIDTH // BLOCK_SIZE or ny >= HEIGHT // BLOCK_SIZE or (ny >= 0 and self.board[ny][nx]):
                        return True
        return False

    def lock_piece(self):
        for y, row in enumerate(self.shape):
            for x, cell in enumerate(row):
                if cell and self.y + y >= 0:
                    self.board[self.y + y][self.x + x] = self.color
        self.clear_lines()
        self.new_piece()

    def clear_lines(self):
        new_board = [row for row in self.board if any(cell == 0 for cell in row)]
        lines_cleared = len(self.board) - len(new_board)

        
        if lines_cleared > 0:
            self.score += lines_cleared * 10

        for _ in range(lines_cleared):
            new_board.insert(0, [0] * (WIDTH // BLOCK_SIZE))
        self.board = new_board

    def move(self, dx, dy):
        if not self.check_collision(dx, dy):
            self.x += dx
            self.y += dy
        elif dy > 0:
            self.lock_piece()

    def rotate(self):
        rotated = [list(row) for row in zip(*self.shape[::-1])]
        if not self.check_collision(shape=rotated):
            self.shape = rotated

def main():
    clock = pygame.time.Clock()
    game = Tetris()
    fall_time = 0

    font = pygame.font.SysFont(None, 36)
    title_font = pygame.font.SysFont(None, 48)

    game_started = False # Flag to control game start/pause state

    button_width = 150
    button_height = 60
    start_button_rect = pygame.Rect(
        (WIDTH // 2) - (button_width // 2),
        (TOP_MARGIN + (HEIGHT - TOP_MARGIN) // 2) - (button_height // 2),
        button_width,
        button_height
    )
    start_button_color = (0, 150, 0)
    start_text_color = WHITE

    while True:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return

            if event.type == pygame.MOUSEBUTTONDOWN:
                if start_button_rect.collidepoint(event.pos):
                    # If game is not started (first launch) OR game is over,
                    # clicking the button will start a new game.
                    if not game_started or game.game_over:
                        game_started = True
                        game = Tetris() # Re-initialize game state
                        fall_time = 0
                        
            if game_started and not game.game_over: # Only process game controls if game is actively playing
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_LEFT:
                        game.move(-1, 0)
                    if event.key == pygame.K_RIGHT:
                        game.move(1, 0)
                    if event.key == pygame.K_DOWN:
                        game.move(0, 1)
                    if event.key == pygame.K_UP:
                        game.rotate()
        
        screen.fill(BLACK) # Clear the screen at the beginning of each frame

        if not game_started: # Initial screen or after game over, before clicking start
            # Draw game title
            title_text = title_font.render("tetris", True, WHITE)
            title_rect = title_text.get_rect(center=(WIDTH // 2, (HEIGHT + TOP_MARGIN) // 4))
            screen.blit(title_text, title_rect)

            # Draw Start button
            pygame.draw.rect(screen, start_button_color, start_button_rect)
            start_text_surface = font.render("game start", True, start_text_color)
            start_text_rect = start_text_surface.get_rect(center=start_button_rect.center)
            screen.blit(start_text_surface, start_text_rect)
        
        elif game.game_over: # Game is over, display game over screen
            game_over_text = title_font.render("game over!", True, (255, 0, 0))
            game_over_rect = game_over_text.get_rect(center=(WIDTH // 2, (HEIGHT + TOP_MARGIN) // 2 - 30))
            screen.blit(game_over_text, game_over_rect)

            final_score_text = font.render(f"score: {game.score}", True, WHITE)
            final_score_rect = final_score_text.get_rect(center=(WIDTH // 2, (HEIGHT + TOP_MARGIN) // 2 + 20))
            screen.blit(final_score_text, final_score_rect)

            # Draw Restart button (using the same button rect)
            pygame.draw.rect(screen, start_button_color, start_button_rect)
            restart_button_text = font.render("restart", True, start_text_color)
            restart_button_rect = restart_button_text.get_rect(center=start_button_rect.center)
            screen.blit(restart_button_text, restart_button_rect)

        else: 
            fall_speed = 500
            fall_time += clock.get_rawtime()
            clock.tick()

            if fall_time >= fall_speed:
                game.move(0, 1)
                fall_time = 0

            
            for x in range(0, WIDTH, BLOCK_SIZE):
                pygame.draw.line(screen, GRAY, (x, TOP_MARGIN), (x, HEIGHT + TOP_MARGIN))
            
            
            pygame.draw.line(screen, WHITE, (0, TOP_MARGIN), (WIDTH, TOP_MARGIN), 2)
            
           
            score_text = font.render(f"score: ({game.score})", True, WHITE)
            screen.blit(score_text, (10, 10))

           
            for y, row in enumerate(game.board):
                for x, cell in enumerate(row):
                    if cell:
                        pygame.draw.rect(screen, COLORS[cell-1], (x * BLOCK_SIZE, y * BLOCK_SIZE + TOP_MARGIN, BLOCK_SIZE, BLOCK_SIZE))
                        pygame.draw.rect(screen, WHITE, (x * BLOCK_SIZE, y * BLOCK_SIZE + TOP_MARGIN, BLOCK_SIZE, BLOCK_SIZE), 1)

            
            for y, row in enumerate(game.shape):
                for x, cell in enumerate(row):
                    if cell:
                        pygame.draw.rect(screen, COLORS[game.color-1], ((game.x + x) * BLOCK_SIZE, (game.y + y) * BLOCK_SIZE + TOP_MARGIN, BLOCK_SIZE, BLOCK_SIZE))
                        pygame.draw.rect(screen, WHITE, ((game.x + x) * BLOCK_SIZE, (game.y + y) * BLOCK_SIZE + TOP_MARGIN, BLOCK_SIZE, BLOCK_SIZE), 1)

        pygame.display.flip()

if __name__ == "__main__":
    main()

#x

/opt/anaconda3/envs/AI/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


2026-03-16 14:09:10.775 python[57828:8038825] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit
